Notebook utilisé pendant le stage pour exporter les images pré-traitées NISAR utilisées en entrée du modèle, à partir des images de backscatter et de cohérence de phase diffusées par la mission NISAR et téléchargées via la plateforme ASF de l'université d'Alaska.  
Les pré-traitements sont les suivants : 
- images backscatter
    - application d'un filtre de quegan avec moyennage spatial 5x5 pour les images descendante et 7x7 pour les ascendantes (moins d'images ascendantes)
    - export des moyennes, std, min, max pour chaque orbite et polarisation (après calcul VV/VH et conversion dB)
- images cohérence de phase
    - sélection des images issues d'acquisition à 12 jours d'intervalle (période normale de revisite de NISAR)
    - export mean, std, min, max pour chaque orbite

In [1]:
import numpy as np
import h5py
import rasterio
import shreklib
import matplotlib.pyplot as plt
import os
from os.path import join
import glob
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
from torch.nn.functional import avg_pool2d
from affine import Affine
from rasterio.warp import reproject
from rasterio.enums import Resampling
import torch
torch.no_grad()
import joblib
from datetime import datetime
from concurrent.futures import ProcessPoolExecutor

/usr/local/lib/python3.12/dist-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
# dossier contenant les images téléchargées sur asf
data_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/backscatter"
fps=glob.glob(join(data_folder, "*.h5"))

In [10]:
def explore(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(f"{name} | shape={obj.shape} | dtype={obj.dtype}")
    else:
        print(f"{name}/")

1) on crée un objet h5py
2) on calcule sa transformation
3) on déduit la bbox de l'image couvrant la hg et on la charge
4) on reprojette

In [100]:
hg=gpd.read_file("/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/emprise_haute_garonne/haute_garonne.shp")

x_coords_fp="science/LSAR/GCOV/grids/frequencyA/xCoordinates"
y_coords_fp="science/LSAR/GCOV/grids/frequencyA/yCoordinates"
x_spacing_fp="science/LSAR/GCOV/grids/frequencyA/xCoordinateSpacing"
y_spacing_fp="science/LSAR/GCOV/grids/frequencyA/yCoordinateSpacing"
crs_fp="science/LSAR/GCOV/grids/frequencyA/projection"

hh_fp="science/LSAR/GCOV/grids/frequencyA/HHHH"
hv_fp="science/LSAR/GCOV/grids/frequencyA/HVHV"

In [48]:
from rasterio.windows import from_bounds

# fonction d'export d'une image geotif de la rétrodiffusion, projeté sur grille de ref, à partir du fichier h5 fourni par asf
def nisar_to_grid(fp):

    # crop de l'image à la Haute-Garonne
    with h5py.File(fp, "r") as f:
        x_spacing=f[x_spacing_fp][()]
        y_spacing=f[y_spacing_fp][()]
        x0=f[x_coords_fp][0]
        y0=f[y_coords_fp][0]
        tile_width=f[x_coords_fp].shape[0]-1
        tile_height=f[y_coords_fp].shape[0]-1
        crs=f[crs_fp][()]

    transform=np.array([[x_spacing, 0, x0],[0, y_spacing, y0], [0,0,1]])
    transform_obj=Affine(*transform.ravel()[:6])
    
    hg_window=from_bounds(*hg.to_crs(epsg=crs).geometry[0].bounds, transform_obj).round_offsets(op="floor").round_lengths(op="ceil") # on arrondi au plus englobant, pour ne pas tronquer de haute-garonne
    col_off, row_off, width, height=hg_window.col_off, hg_window.row_off, hg_window.width, hg_window.height

    col_off = max(0, hg_window.col_off)
    row_off = max(0, hg_window.row_off)

    width = min(tile_width, hg_window.width - (col_off - hg_window.col_off))
    height = min(tile_height, hg_window.height - (row_off - hg_window.row_off))

    hg_window=rasterio.windows.Window(col_off, row_off, width, height)
    crop_transform=rasterio.windows.transform(hg_window, transform_obj)

    with h5py.File(fp, "r") as f:
        hh=f[hh_fp][row_off:row_off+height, col_off:col_off+width]
        hv=f[hv_fp][row_off:row_off+height, col_off:col_off+width]

    # reprojection dans la grille de ref
    out_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/backscatter_snapped_to_grid"    
    ref_transform=Affine(*shreklib.ref_grid["transform"].ravel()[:6])

    out_name=os.path.basename(fp)[:-3]+".tif"
    out_fp=join(out_folder, out_name)
    profile={
        "crs":32631,
        "transform":ref_transform,
        "width":shreklib.ref_grid["width"],
        "height":shreklib.ref_grid["height"],
        "nodata":np.nan,
        "dtype":"float32", 
        "count":2
    }
    with rasterio.open(out_fp, "w", **profile) as dst:
        for i, (name, band) in enumerate(zip(["hh", "hv"], [hh, hv])):
            reproject(
                source=band, #rasterio.band donne directement les instructions à gdal et ne charge pas tout en mémoire python.
                destination=rasterio.band(dst,i+1),

                src_transform=crop_transform,
                dst_transform=ref_transform,

                src_crs=crs.item(),
                dst_crs=32631,

                dst_nodata=np.nan,
                resampling=Resampling.bilinear
            )
            dst.set_band_description(i+1, name)
            
shreklib.batch_map(nisar_to_grid, fps)

# ajout du fichier non traité par batch map pour bug inconnu

In [160]:
l1=list(map(lambda fp:os.path.basename(fp)[:-3],fps))
l2=list(map(lambda fp:os.path.basename(fp)[:-4],glob.glob(join(out_folder, "*.tif"))))
not_processed=[]
for i, fp in enumerate(l1):
    if fp not in l2:
        not_processed.append((i, fp))
for i, fp in not_processed:
    nisar_to_grid(fps[i])

# filtre de Quegan

In [21]:
# on distingue bien les images ascendantes et descendantes, que l'on ne filtre pas ensemble
reproj_folder='/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/backscatter_snapped_to_grid'
reproj_fps=glob.glob(join(reproj_folder, "*.tif"))
parse_orbit=lambda fp:os.path.basename(fp)[:-4].split("_")[6]
fps_df=pd.DataFrame({"fps":reproj_fps, "orbit":list(map(parse_orbit, reproj_fps))})
descending_fps=fps_df.loc[fps_df.orbit=="D"].fps.tolist()
descending_fps=fps_df.loc[fps_df.orbit=="D"].fps.tolist()
descending=torch.empty((len(descending_fps), 2, shreklib.ref_grid["height"], shreklib.ref_grid["width"]), dtype=torch.float32)
for i, fp in tqdm(enumerate(descending_fps)):
    with rasterio.open(fp, "r") as src:
        descending[i]=torch.from_numpy(src.read())
descending_hh=descending[:,0,:,:]
full_descending_hh_nei=avg_pool2d(
    torch.nan_to_num(descending_hh, nan=0),
    kernel_size=5,
    stride=1,
    padding=2
)

valid_nei=avg_pool2d(
    torch.isfinite(descending_hh).float(),
    kernel_size=5,
    stride=1,
    padding=2
)

nanned_descending_hh_nei=full_descending_hh_nei/valid_nei.clamp_min(1e-12)
descending_hh_queganed=nanned_descending_hh_nei*torch.nanmean(descending_hh/nanned_descending_hh_nei, dim=0, keepdims=True)
temp_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/torch_queganed"
torch.save(descending_hh_queganed,join(temp_folder, "descending_hh"))
joblib.dump(descending_fps, join(temp_folder, "descending_hh_filenames.pkl"))

# fonction pour appliquer quegan
On applique quegan par orbite pour la cohérence radio et par polarisation pour ne pas saturer la ram en traitant les deux polarisations d'un coup

In [3]:
# fonction pour appliquer le filtre de quegan
def quegan(fps, pol, kernel_sizes=[5]): #pol est l'indice à lire des fichiers tifs snapped to grid. Donc 1 pour HH et 2 pour HV.

    nisar_imgs=torch.empty((len(fps), shreklib.ref_grid["height"], shreklib.ref_grid["width"]), dtype=torch.float32)
    for i, fp in tqdm(enumerate(fps)):
        with rasterio.open(fp, "r") as src:
            nisar_imgs[i]=torch.from_numpy(src.read(pol))

    queganed_list=[]
    for kernel_size in kernel_sizes:
        padding=kernel_size//2
        nisar_imgs_nei=avg_pool2d(
            torch.nan_to_num(nisar_imgs, nan=0),
            kernel_size=kernel_size,
            stride=1,
            padding=padding
        )

        valid_nei=avg_pool2d(
            torch.isfinite(nisar_imgs).float(),
            kernel_size=kernel_size,
            stride=1,
            padding=padding
        )

        nanned_nisar_imgs_nei=nisar_imgs_nei/valid_nei.clamp_min(1e-12)
        queganed=nanned_nisar_imgs_nei*torch.nanmean(nisar_imgs/nanned_nisar_imgs_nei, dim=0, keepdims=True)
        queganed_list.append(torch.where(torch.isnan(nisar_imgs), torch.nan, queganed))
    return queganed_list

In [2]:
# on distingue bien les images ascendantes et descendantes, que l'on ne filtre pas ensemble
#exports
reproj_folder='/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/backscatter_snapped_to_grid'
reproj_fps=glob.glob(join(reproj_folder, "*.tif"))
parse_orbit=lambda fp:os.path.basename(fp)[:-4].split("_")[6]
fps_df=pd.DataFrame({"fps":reproj_fps, "orbit":list(map(parse_orbit, reproj_fps))})
descending_fps=fps_df.loc[fps_df.orbit=="D"].fps.tolist()
ascending_fps=fps_df.loc[fps_df.orbit=="A"].fps.tolist()
descending_hh_queganed=quegan(descending_fps, 1)
descending_hv_queganed=quegan(descending_fps, 2)
ascending_hh_queganed=quegan(ascending_fps, 1, kernel_sizes=[7, 11, 15])
ascending_hv_queganed=quegan(ascending_fps, 2, kernel_sizes=[7, 11, 15])

temp_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/torch_queganed"
torch.save(descending_hh_queganed, join(temp_folder, "descending_hh_queganed"))
torch.save(descending_hv_queganed, join(temp_folder, "descending_hv_queganed"))

temp_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/torch_queganed"
descending_hh_queganed=torch.load(join(temp_folder, "descending_hh_queganed"))
descending_hv_queganed=torch.load(join(temp_folder, "descending_hv_queganed"))

temp_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/torch_queganed"
torch.save(ascending_hh_queganed, join(temp_folder, "ascending_hh_queganed"))
torch.save(ascending_hv_queganed, join(temp_folder, "ascending_hv_queganed"))

temp_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/torch_queganed"
joblib.dump(ascending_hh_queganed, join(temp_folder, "ascending_hh_queganed_multi_kernel.pkl"))
joblib.dump(ascending_hv_queganed, join(temp_folder, "ascending_hv_queganed_multi_kernel.pkl"))

temp_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/torch_queganed"
ascending_hh_queganed=joblib.load(join(temp_folder, "ascending_hh_queganed_multi_kernel.pkl"))
ascending_hv_queganed=joblib.load(join(temp_folder, "ascending_hv_queganed_multi_kernel.pkl"))

temp_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/torch_queganed"
ascending_hv_queganed=joblib.load(join(temp_folder, "ascending_hv_queganed_multi_kernel.pkl"))

temp_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/torch_queganed"
ascending_hh_queganed=torch.load(join(temp_folder, "ascending_hh_queganed"))
ascending_hv_queganed=torch.load(join(temp_folder, "ascending_hv_queganed"))

# export stats

In [3]:
# fonction pour exporter mean, std, min, max
def export_stats(nisar_imgs, out_path, db=True):
    mean=torch.nanmean(nisar_imgs, dim=0)
    std=torch.sqrt(torch.nanmean(nisar_imgs**2, dim=0)-mean**2)
    min=torch.min(torch.nan_to_num(nisar_imgs, nan=torch.inf), dim=0).values
    max=torch.max(torch.nan_to_num(nisar_imgs, nan=-torch.inf), dim=0).values
    if db==True:
        shreklib.tif_export(10*torch.log10(torch.stack([mean, std, min, max])).numpy(), shreklib.ref_grid["transform"], 32631, out_path, dtype="float32")
    else:
        shreklib.tif_export(torch.stack([mean, std, min, max]).numpy(), shreklib.ref_grid["transform"], 32631, out_path, dtype="float32")

In [17]:
export_stats(ascending_hh_queganed[2], "/mnt/RAM_disk/ascending_hh_queganed15.tif")
export_stats(ascending_hv_queganed[0], "/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/nisar_stats/ascending_hv_queganed7_stats.tif")
export_stats(ascending_hh_queganed[0]/ascending_hv_queganed[0], "/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/nisar_stats/ascending_hh_hv_queganed7_stats.tif")
export_stats(descending_hh_queganed, "/mnt/RAM_disk/nisar_d_hh_stats.tif")
export_stats(descending_hv_queganed, "/mnt/RAM_disk/nisar_d_hv_stats.tif")
export_stats(descending_hh_queganed/descending_hv_queganed, "/mnt/RAM_disk/nisar_d_hh_hv_stats.tif")
shreklib.tif_export(10*torch.log10(torch.stack([torch.nanmean(descending_hh_queganed, dim=0),
torch.nanmean(descending_hv_queganed, dim=0),
torch.nanmean(descending_hh_queganed/descending_hv_queganed, dim=0)])), shreklib.ref_grid["transform"], 32631, "/mnt/RAM_disk/nisar_descending_means.tif", dtype="float32")
export_stats(ascending_hh_queganed, "/mnt/RAM_disk/nisar_a_hh_stats.tif")
export_stats(ascending_hv_queganed, "/mnt/RAM_disk/nisar_a_hv_stats.tif")
export_stats(ascending_hh_queganed/ascending_hv_queganed, "/mnt/RAM_disk/nisar_a_hh_hv_stats.tif")

# interférogrammes

In [4]:
data_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/interferograms"
interferograms_fps=glob.glob(join(data_folder, "*.h5"))
parse_first_date= lambda fp:datetime.strptime(os.path.basename(fp)[:-4].split("_")[11].split("T")[0],r"%Y%m%d")
parse_second_date= lambda fp:datetime.strptime(os.path.basename(fp)[:-4].split("_")[13].split("T")[0],r"%Y%m%d")
interferograms_df=pd.DataFrame({"fps":interferograms_fps, 
"first_date":list(map(parse_first_date, interferograms_fps)), 
"second_date":list(map(parse_second_date, interferograms_fps))})
interferograms_df["time_delta"]=interferograms_df.second_date-interferograms_df.first_date
# on ne garde que les images de cohérence calculées sur des acquisitions à 12 jours d'intervalle
from datetime import timedelta
interferograms_12=interferograms_df.loc[interferograms_df.time_delta<=timedelta(days=12)]
fps12=interferograms_12.fps.tolist()

# reprojection des images de cohérence

In [75]:
from affine import Affine
from rasterio.warp import reproject
from rasterio.enums import Resampling

hg=gpd.read_file("/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/emprise_haute_garonne/haute_garonne.shp")

x_coords_fp="science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram/HH/xCoordinates"
y_coords_fp="science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram/HH/yCoordinates"
x_spacing_fp="science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram/HH/xCoordinateSpacing"
y_spacing_fp="science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram/HH/yCoordinateSpacing"
crs_fp="science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram/projection"

coherence_fp="science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram/HH/coherenceMagnitude"
from rasterio.windows import from_bounds

hg=gpd.read_file("/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/emprise_haute_garonne/haute_garonne.shp")

# reprojectiond des images de cohérence à la grille de référence et export en .tif
def coherence_to_grid(fp):

    x_coords_fp="science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram/HH/xCoordinates"
    y_coords_fp="science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram/HH/yCoordinates"
    x_spacing_fp="science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram/HH/xCoordinateSpacing"
    y_spacing_fp="science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram/HH/yCoordinateSpacing"
    crs_fp="science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram/projection"

    coherence_fp="science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram/HH/coherenceMagnitude"

    # crop de l'image à la Haute-Garonne
    with h5py.File(fp, "r") as f:
        x_spacing=f[x_spacing_fp][()]
        y_spacing=f[y_spacing_fp][()]
        x0=f[x_coords_fp][0]
        y0=f[y_coords_fp][0]
        tile_width=f[x_coords_fp].shape[0]-1
        tile_height=f[y_coords_fp].shape[0]-1
        crs=f[crs_fp][()]

    transform=np.array([[x_spacing, 0, x0],[0, y_spacing, y0], [0,0,1]])
    transform_obj=Affine(*transform.ravel()[:6])
    
    hg_window=from_bounds(*hg.to_crs(epsg=crs).geometry[0].bounds, transform_obj).round_offsets(op="floor").round_lengths(op="ceil") # on arrondi au plus englobant, pour ne pas tronquer de haute-garonne
    col_off, row_off, width, height=hg_window.col_off, hg_window.row_off, hg_window.width, hg_window.height

    col_off = max(0, hg_window.col_off)
    row_off = max(0, hg_window.row_off)

    width = min(tile_width, hg_window.width - (col_off - hg_window.col_off))
    height = min(tile_height, hg_window.height - (row_off - hg_window.row_off))

    hg_window=rasterio.windows.Window(col_off, row_off, width, height)
    crop_transform=rasterio.windows.transform(hg_window, transform_obj)

    with h5py.File(fp, "r") as f:
        coherence=f[coherence_fp][row_off:row_off+height, col_off:col_off+width]

    # reprojection dans la grille de ref
    out_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/coherence_snapped_to_grid"
    if not os.path.exists(out_folder):
        os.mkdir(out_folder)

    ref_transform=Affine(*shreklib.ref_grid["transform"].ravel()[:6])

    out_name=os.path.basename(fp)[:-3]+".tif"
    out_fp=join(out_folder, out_name)
    profile={
        "crs":32631,
        "transform":ref_transform,
        "width":shreklib.ref_grid["width"],
        "height":shreklib.ref_grid["height"],
        "nodata":np.nan,
        "dtype":"float32", 
        "count":1
    }
    with rasterio.open(out_fp, "w", **profile) as dst:
        name, band="coherence", coherence
        reproject( 
            source=band, #rasterio.band donne directement les instructions à gdal et ne charge pas tout en mémoire python.
            destination=rasterio.band(dst,1),

            src_transform=crop_transform,
            dst_transform=ref_transform,

            src_crs=crs.item(),
            dst_crs=32631,

            dst_nodata=np.nan,
            resampling=Resampling.bilinear
        )
        dst.set_band_description(1, name)

In [ ]:
with ProcessPoolExecutor(max_workers=32) as executor:
    list(executor.map(coherence_to_grid, fps12))
l1=list(map(lambda fp:os.path.basename(fp)[:-3],fps12))
out_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/coherence_snapped_to_grid"
l2=list(map(lambda fp:os.path.basename(fp)[:-4],glob.glob(join(out_folder, "*.tif"))))
not_processed=[]
for i, fp in enumerate(l1):
    if fp not in l2:
        not_processed.append((i, fp))
for i, fp in not_processed:
    nisar_to_grid(fps[i])

# coherence stats

In [3]:
out_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/coherence_snapped_to_grid"
coherence_snapped_fps=glob.glob(join(out_folder, "*.tif"))

In [ ]:
# fonction analogue à export_stats mais sans utiliser torch, qui semble faire planter le kernel de façon un peu aléatoire
def export_stats_numpy(nisar_imgs, out_path, db=True):
    mean=np.nanmean(nisar_imgs, axis=0)
    std=np.sqrt(np.nanmean(nisar_imgs**2, axis=0)-mean**2)
    min=np.min(np.nan_to_num(nisar_imgs, nan=np.inf), axis=0)
    max=np.max(np.nan_to_num(nisar_imgs, nan=-np.inf), axis=0)
    if db==True:
        shreklib.tif_export(10*np.log10(np.stack([mean, std, min, max])).numpy(), shreklib.ref_grid["transform"], 32631, out_path, dtype="float32")
    else:
        shreklib.tif_export(np.stack([mean, std, min, max]), shreklib.ref_grid["transform"], 32631, out_path, dtype="float32")

In [5]:
coherences=torch.empty((len(coherence_snapped_fps), shreklib.ref_grid["height"], shreklib.ref_grid["width"]), dtype=torch.float32)
for i,fp in tqdm(enumerate(coherence_snapped_fps)):
    with rasterio.open(fp, "r") as src:
        coherences[i]=src.read(1)
orbits=np.array(list(map(parse_orbit, coherence_snapped_fps)))
coherence_stats_folder="/mnt/Data/30_Stages_Encours/2026/2026_SHREQ_Niels/02_Donnees/input/NISAR/coherence_stats"
export_stats_numpy(coherences[orbits=='A'], join(coherence_stats_folder,"nisar_ascending_coherences_stats.tif"), db=False)

export_stats_numpy(coherences[orbits=='D'], join(coherence_stats_folder,"nisar_descending_coherences_stats.tif"), db=False)